# 詩織 LoRA v4f テスト生成ノートブック（Kaggle） — IP-Adapter参照顔テスト

学習済みLoRA（train-shiori-lora-kaggle-v4 Version 2 の出力 shiori_oneesan_v4.safetensors）を **Add Input** で接続して使う。学習は行わない。

狙い：LoRAの「記憶」に頼らず、生成時に参照写真そのものを見ながら顔を描く **IP-Adapter（Plus版）** を併用し、目の形・輪郭が参照に寄るかを見る。
- ベースは LoRA の母体 Animagine XL 4.0、LoRA scale 1.0、プロンプト・シード（42）・28ステップは v4 と同一
- 参照画像は学習データの 00_closeup_window.png（GitHubのデータセットzipから取得）
- IP-Adapter の強さ 0.5 / 0.8 の2段階 × 2場面 = 4枚

In [ ]:
# GPU確認
!nvidia-smi


In [ ]:
# 依存関係の用意（torchao は古い版が入っていると peft が ImportError になるため外す）
!pip install -q diffusers transformers accelerate peft huggingface_hub
!pip uninstall -y torchao

# ベースモデル（Animagine XL 4.0）を取得
from huggingface_hub import hf_hub_download
import shutil, os
p = hf_hub_download(repo_id='cagliostrolab/animagine-xl-4.0', filename='animagine-xl-4.0.safetensors')
shutil.copy(p, '/kaggle/working/animagine-xl-4.0.safetensors')
print('model ready:', os.path.getsize('/kaggle/working/animagine-xl-4.0.safetensors'))

# 参照画像を用意（GitHubのデータセットzipから 00_closeup_window.png を取り出す）
!git clone --depth 1 https://github.com/keen-fox-24/shiori-sd.git /kaggle/working/shiori-sd
!cd /kaggle/working/shiori-sd && unzip -o -q shiori-dataset-v4.zip -d /kaggle/working/dataset-v4
import glob
refs = glob.glob('/kaggle/working/dataset-v4/**/00_closeup_window.png', recursive=True)
print('reference candidates:', refs)
assert refs, '00_closeup_window.png not found in dataset zip'
REF_PATH = refs[0]


In [ ]:
# LoRA + IP-Adapter Plus を読み込んでテスト生成（参照顔あり、強さ2段階 × 2場面 = 4枚）
import glob, os, torch
from PIL import Image
from transformers import CLIPVisionModelWithProjection
from diffusers import StableDiffusionXLPipeline
cands = [c for c in glob.glob('/kaggle/input/**/shiori_oneesan_v4.safetensors', recursive=True)]
print('lora candidates:', cands)
assert cands, 'shiori_oneesan_v4.safetensors not found in /kaggle/input'
lora_path = cands[0]
print('lora size:', os.path.getsize(lora_path))
ref_image = Image.open(REF_PATH).convert('RGB')
print('reference image:', REF_PATH, ref_image.size)
image_encoder = CLIPVisionModelWithProjection.from_pretrained('h94/IP-Adapter', subfolder='models/image_encoder', torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_single_file('/kaggle/working/animagine-xl-4.0.safetensors', torch_dtype=torch.float16, image_encoder=image_encoder).to('cuda')
pipe.load_lora_weights(os.path.dirname(lora_path), weight_name='shiori_oneesan_v4.safetensors')
pipe.load_ip_adapter('h94/IP-Adapter', subfolder='sdxl_models', weight_name='ip-adapter-plus_sdxl_vit-h.bin')
neg = 'lowres, bad anatomy, bad hands, extra fingers, worst quality, blurry, multiple girls, 2girls, extra person, tsurime, sharp eyes, narrow eyes'
os.makedirs('/kaggle/working/output', exist_ok=True)
prompts = [
    ('outing', 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, side part, purple eyes, gentle eyes, droopy eyes, tareme, gentle smile, soft expression, pink cardigan, shopping street, cafe terrace, sunny day'),
    ('hair', 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, side part, purple eyes, gentle eyes, droopy eyes, tareme, gentle smile, soft expression, hair over shoulder, white blouse, indoors, window, daylight, close-up portrait'),
]
for tag, prompt in prompts:
    for scale in (0.5, 0.8):
        pipe.set_ip_adapter_scale(scale)
        g = torch.Generator(device='cuda').manual_seed(42)
        img = pipe(prompt, negative_prompt=neg, ip_adapter_image=ref_image, width=1024, height=1024, num_inference_steps=28, guidance_scale=7, generator=g).images[0]
        name = 'test_%s_v4f_ipa%s.png' % (tag, str(scale).replace('.', ''))
        img.save('/kaggle/working/output/' + name)
        print('saved', name)
print('all done')
